# Bài tập Điểm cộng: Xây dựng Complex Data Generator
## (Dựa trên toàn bộ EDA Tuần 1 — 6 Notebooks)

Đây là bộ mô phỏng dữ liệu (Data Simulator) hoàn chỉnh, kế thừa **đầy đủ** các tham số từ kết quả EDA Tuần 1:

| Notebook | Chủ đề | Insight Tham chiếu |
|---|---|---|
| NB 1 | Làm sạch dữ liệu | Cấu trúc dữ liệu, loại lỗi thực tế |
| NB 2 | Phân tích thời gian | Giờ cao điểm (7-9h, 17-19h), pattern ngày trong tuần |
| NB 3 | Phân tích địa lý | Phân bổ Urban/Suburban (Manhattan vs Outer Boroughs) |
| NB 4 | Hành vi & giá cước | Avg fare=$17.60, Median=$13.50, fare/mile=$8.43 |
| NB 5 | Feature Engineering | Tổng hợp đặc trưng từ NB 1-4 |
| NB 6 | Kaggle Ridesharing EDA | rides/user~Poisson(5), fare~2.5USD/km, age~Normal(30,8) |

### Giả định Mô phỏng (Simulation Assumptions): Chuyển đổi Level of Analysis
Một điểm khác biệt lớn giữa 2 bộ dữ liệu EDA là: **Kaggle có sẵn User-level data**, trong khi **Yellow Taxi chỉ có Trip-level data**.
Để tích hợp insight từ Yellow Taxi vào bộ mô phỏng User-level này, chúng ta sử dụng phương pháp **Parametric Simulation** (Mô phỏng tham số) với các giả định sau:
1. **Doanh thu (Fare):** Tổng doanh thu của 1 user = `số chuyến (rides) × doanh thu trung bình 1 chuyến (avg_fare_per_trip)`. Giá trị trung bình được cấu hình sao cho sát với $17.60 (insight từ NB4).
2. **Hành vi gọi xe (Preferred Hour):** Mỗi user được gán một khung giờ gọi xe ưa thích, lấy mẫu theo trọng số dựa trên phân phối giờ cao điểm của toàn bộ chuyến xe (insight từ NB2). Bằng cách này, tập hợp (aggregate) nhu cầu của 20,000 users sẽ tái tạo lại chính xác pattern giờ cao điểm của toàn thành phố.
3. **Phân bổ địa lý (Urban vs Suburban):** Dựa vào insight >80% chuyến xe xuất phát từ Manhattan (Urban Zone) trong NB3, ta giả định ~60% lượng User sống hoặc hoạt động chủ yếu ở khu vực Đô thị (Urban).

Bộ dữ liệu này sẽ được dùng cho:
- **Tuần 5:** Kiểm tra Covariate Balance trong A/A Test.
- **Tuần 6:** Thực hành Observational Studies (Matching, Adjustment).

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_theme(style='whitegrid')
print('Libraries loaded successfully.')

## 1. Tham số nền từ EDA Thực tế

**Từ NB 4 (Behavior Analysis — Yellow Taxi thực tế):**
- Avg Fare: $17.60
- Median Fare: $13.50
- Avg Fare/mile: $8.43/mile (~$5.24/km)
- Median Fare/mile: $7.50/mile (~$4.66/km)

**Từ NB 6 (Kaggle Ridesharing EDA):**
- rides/user ~ Poisson(lambda=5) — phân phối hình chuông, đỉnh ở 4-6 chuyến
- fare ≈ 2.5 USD/km (r=0.93) — tương quan cực mạnh
- age ~ Normal(loc=30, scale=8), clip(18, 60)
- gender: Male 55%, Female 40%, Other 5%
- driver_rating ~ Normal(loc=4.5, scale=0.3) — Positivity Bias

**Từ NB 2 (Time Pattern Analysis — Yellow Taxi):**
- Giờ cao điểm sáng: 7h-9h (nhu cầu gấp ~3x giờ thấp điểm)
- Giờ cao điểm chiều: 17h-19h (nhu cầu gấp ~3x giờ thấp điểm)
- Pattern ngày: Thứ 6-7 là bận nhất

**Từ NB 3 (Geo Pattern Analysis — Yellow Taxi):**
- Manhattan (Urban Zone): Chiếm >80% chuyến đi
- Outer Boroughs (Suburban): Chiếm ~20% chuyến đi
- Khách Urban: Đi xe nhiều hơn, nhạy cảm hơn với promotion

In [ ]:
# Tham số vũ trụ
N_USERS = 20000
TRUE_ATE = 0.85  # Voucher thực sự làm tăng +0.5 chuyến/người

# === TỪ NB 6 (Kaggle Ridesharing EDA) ===
RIDES_LAMBDA = 5         # rides/user ~ Poisson(lam=5)
FARE_PER_KM_KAGGLE = 2.5  # Kaggle: fare ≈ 2.5 USD/km (r=0.93)
AGE_MEAN = 30
AGE_STD = 8
GENDER_PROBS = [0.55, 0.40, 0.05]  # Male, Female, Other
DRIVER_RATING_MEAN = 4.5
DRIVER_RATING_STD = 0.3

# === TỪ NB 4 (Yellow Taxi Behavior Analysis) ===
AVG_FARE_USD = 17.60       # Giá cước trung bình
MEDIAN_FARE_USD = 13.50    # Giá cước trung vị
AVG_FARE_PER_MILE = 8.43   # ≈ 5.24 USD/km

# === TỪ NB 2 (Yellow Taxi Time Analysis) ===
# Hệ số nhân nhu cầu theo giờ (0-23)
HOUR_DEMAND_MULTIPLIER = [
    0.3, 0.2, 0.2, 0.15, 0.15, 0.2,  # 0h-5h (đêm khuya, ít người)
    0.5, 1.0, 1.5,                    # 6h-8h (cao điểm sáng bắt đầu)
    1.5,                              # 9h (cao điểm sáng)
    1.0, 0.8, 0.7, 0.7,              # 10h-13h (giờ hành chính)
    0.8, 0.9,                         # 14h-15h
    1.0, 1.5, 1.5,                    # 16h-18h (cao điểm chiều)
    1.3, 1.2, 1.0,                    # 19h-21h (tối)
    0.8, 0.5                          # 22h-23h (đêm)
]

# === TỪ NB 3 (Yellow Taxi Geo Analysis) ===
URBAN_RATIO = 0.60  # 60% khách sống ở đô thị (Urban Zone)

print(f'Config loaded: {N_USERS:,} users | True ATE = {TRUE_ATE} chuyến')

## 2. Lớp DataGenerator Hoàn chỉnh

In [ ]:
class RideSharingDataGenerator:
    """
    Sinh bộ dữ liệu mô phỏng chân thực cho bài toán Causal Inference.
    Toàn bộ tham số được hiệu chỉnh dựa trên EDA Tuần 1 (6 notebooks).

    Covariates (Đặc trưng người dùng):
    - age            : Tuổi người dùng. Nguồn NB6 - Normal(30, 8).
    - gender         : Giới tính. Nguồn NB6 - [Male 55%, Female 40%, Other 5%].
    - is_urban       : Có sống ở khu đô thị không. Nguồn NB3 - Urban ~60%.
    - recency_days   : Số ngày không mở app. Dựa trên Experiment Design v2.
    - monthly_rides  : Số chuyến/tháng (lịch sử). Nguồn NB6 - Poisson(5).
    - preferred_hour : Giờ gọi xe thường xuyên nhất. Nguồn NB2.
    - customer_seg   : Phân khúc: Occasional(<4), Regular(4-8), Heavy(>8).

    Outcome:
    - y_obs          : Số chuyến thực tế (kịch bản bị nhiễu).
    - y_rand         : Số chuyến thực tế (kịch bản A/B Test chuẩn).
    - fare_obs/rand  : Doanh thu (USD). Nguồn NB4 - Avg $17.60, fare~2.5USD/km.
    """

    def __init__(self, n_users=N_USERS, true_ate=TRUE_ATE):
        self.n = n_users
        self.true_ate = true_ate
        self.df = None

    # === BƯỚC 1: SINH COVARIATES ===

    def _demographics(self):
        """Nhân khẩu học - Nguồn NB6, Section 6
        Kaggle bị random hóa (Uniform 18-65, 33/33/33%) → Tự thiết kế lại theo thực tế:
        - age ~ Normal(30, 8), clip(18, 60)
        - gender: Male 55%, Female 40%, Other 5%
        """
        age = np.clip(
            np.random.normal(AGE_MEAN, AGE_STD, self.n), 18, 60
        ).astype(int)
        gender = np.random.choice(
            ['Male', 'Female', 'Other'], size=self.n, p=GENDER_PROBS
        )
        return age, gender

    def _location(self):
        """Vị trí địa lý - Nguồn NB3 (Geo Pattern Analysis)
        Manhattan Yellow Zone chiếm >80% chuyến đi.
        Khách Urban: nhiều chuyến hơn, nhạy cảm hơn với promotion.
        """
        return np.random.binomial(n=1, p=URBAN_RATIO, size=self.n)

    def _behavior(self, is_urban):
        """Hành vi người dùng - Nguồn NB6 Section 2 + Experiment Design v2
        - Số chuyến/tháng ~ Poisson(5) (NB6: phân phối hình chuông, đỉnh 4-6)
        - Recency (ngày không mở app): Khách Urban active hơn → recency thấp hơn
        - preferred_hour: Dựa trên pattern giờ cao điểm NB2
        """
        # Rides/user/tháng - NB6 Section 2
        monthly_rides = np.random.poisson(lam=RIDES_LAMBDA, size=self.n)

        # Phân khúc khách hàng (Customer Segment)
        segment = np.where(
            monthly_rides < 4, 'Occasional',
            np.where(monthly_rides <= 8, 'Regular', 'Heavy')
        )

        # Recency: Khách Urban tích cực hơn (recency thấp hơn)
        # Experiment Design v2 target: nhóm 5-14 ngày không mở app
        recency_mean = np.where(is_urban == 1, 4, 10)
        recency_days = np.clip(
            np.random.poisson(lam=recency_mean), 0, 30
        ).astype(int)

        # Giờ ưa thích gọi xe (dựa trên NB2 time pattern)
        # Giờ cao điểm sáng (7-9h) và chiều (17-19h) có xác suất cao hơn
        hour_weights = np.array(HOUR_DEMAND_MULTIPLIER)
        hour_probs = hour_weights / hour_weights.sum()
        preferred_hour = np.random.choice(
            range(24), size=self.n, p=hour_probs
        )

        return monthly_rides, segment, recency_days, preferred_hour

    # === BƯỚC 2: TÍNH BASE RIDES (Y0) ===

    def _base_rides(self, monthly_rides, recency_days, is_urban, preferred_hour):
        """Số chuyến tự nhiên (không có voucher).
        Phụ thuộc: Lịch sử rides, Recency, Vị trí, Giờ gọi xe.
        """
        # Hệ số nhu cầu theo giờ (NB2)
        hour_effect = np.array([HOUR_DEMAND_MULTIPLIER[h] for h in preferred_hour])

        base = (
            monthly_rides * 0.5         # Lịch sử rides tác động dương
            - recency_days * 0.15       # Ngủ đông làm giảm chuyến đi
            + is_urban * 1.5            # Khách đô thị đi nhiều hơn
            + (hour_effect - 0.7) * 0.5 # Giờ cao điểm tăng nhu cầu
        )
        noise = np.random.normal(0, 1.2, self.n)
        return np.clip(base + noise, 0, None)

    # === BƯỚC 3: PHÂN BỔ TREATMENT ===

    def _treatment_observed(self, monthly_rides, is_urban):
        """Phân bổ bị nhiễu (Observational Data).
        Hệ thống ưu tiên phát voucher cho khách Urban và khách thường xuyên.
        → Đây là nguồn gây ra Confounding Bias.
        """
        prob_t = 0.1 + (is_urban * 0.35) + (monthly_rides * 0.04)
        prob_t = np.clip(prob_t, 0.05, 0.90)
        return np.random.binomial(1, p=prob_t)

    def _treatment_random(self):
        """Phân bổ ngẫu nhiên (A/B Test - Chuẩn Vàng).
        Tung đồng xu 50/50, không phân biệt đặc điểm người dùng.
        """
        return np.random.binomial(1, p=0.5, size=self.n)

    # === BƯỚC 4: TÍNH OUTCOME ===

    def _outcome(self, base_rides, treatment):
        """Y = Base + True ATE * Treatment + Noise"""
        noise = np.random.normal(0, 0.8, self.n)
        y = base_rides + (self.true_ate * treatment) + noise
        return np.clip(y, 0, None).round().astype(int)

    # === BƯỚC 5: TÍNH DOANH THU ===

    def _fare(self, y_rides, is_urban):
        """Doanh thu (USD) - Nguồn NB4 + NB6.

        NB4 (Yellow Taxi thực tế):
        - Avg Fare = $17.60, Median Fare = $13.50
        - Avg fare/mile = $8.43 (~$5.24/km)

        NB6 (Kaggle): fare ≈ 2.5 USD/km (r=0.93)

        → Kết hợp: Ước tính avg distance ~3.4km (thực tế NYC thường 2-4 dặm),
          dùng $5 USD/km (giữa 2 mức NB4 và NB6) để fare trung bình ~ $17.
        """
        FARE_PER_KM = 5.0  # Kết hợp NB4 ($5.24/km) và NB6 (2.5/km) → 5.0
        # Khách Urban thường đi gần hơn (nội thành), Suburban đi xa hơn
        avg_distance = np.where(
            is_urban == 1,
            np.random.normal(3.0, 1.0, self.n),   # Urban: ~3km/chuyến
            np.random.normal(5.5, 2.0, self.n)    # Suburban: ~5.5km/chuyến
        )
        avg_distance = np.clip(avg_distance, 0.5, 25)
        noise = np.random.normal(0, 2.5, self.n)
        fare = y_rides * (FARE_PER_KM * avg_distance + noise)
        return np.clip(fare, 0, None).round(2)

    # === PIPELINE CHÍNH ===

    def run(self):
        """Sinh và trả về DataFrame đầy đủ."""
        # 1. Covariates
        age, gender = self._demographics()
        is_urban = self._location()
        monthly_rides, segment, recency_days, preferred_hour = self._behavior(is_urban)

        # 2. Base Rides
        base = self._base_rides(monthly_rides, recency_days, is_urban, preferred_hour)

        # 3. Treatment
        t_obs = self._treatment_observed(monthly_rides, is_urban)
        t_rand = self._treatment_random()

        # 4. Outcome
        y_obs = self._outcome(base, t_obs)
        y_rand = self._outcome(base, t_rand)

        # 5. Fare
        fare_obs = self._fare(y_obs, is_urban)
        fare_rand = self._fare(y_rand, is_urban)

        self.df = pd.DataFrame({
            'user_id':           range(1, self.n + 1),
            # --- Covariates ---
            'age':               age,
            'gender':            gender,
            'is_urban':          is_urban,
            'customer_segment':  segment,
            'monthly_rides_history': monthly_rides,
            'recency_days':      recency_days,
            'preferred_hour':    preferred_hour,
            # --- Ẩn (chỉ biết khi mô phỏng) ---
            'base_rides':        base.round(1),
            # --- Kịch bản 1: Observational Data (Bị nhiễu) ---
            'treatment_obs':     t_obs,
            'y_obs':             y_obs,
            'fare_obs':          fare_obs,
            # --- Kịch bản 2: Randomized A/B Test ---
            'treatment_rand':    t_rand,
            'y_rand':            y_rand,
            'fare_rand':         fare_rand,
        })
        return self.df

In [ ]:
# Sinh dữ liệu
gen = RideSharingDataGenerator(n_users=N_USERS, true_ate=TRUE_ATE)
df = gen.run()

print(f'Dataset shape: {df.shape}')
display(df.head(10))

## 3. Kiểm chứng: Bộ dữ liệu có khớp với tham số EDA thực tế không?

So sánh từng tham số quan trọng với insight đã rút ra.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 10))

# 1. Tuổi (NB6)
sns.histplot(df['age'], bins=30, kde=True, ax=axes[0][0], color='steelblue')
axes[0][0].set_title(f'Tuổi\nMean={df["age"].mean():.1f} (EDA: ~30)')
axes[0][0].axvline(df['age'].mean(), color='red', linestyle='--')

# 2. Giới tính (NB6)
gender_counts = df['gender'].value_counts()
axes[0][1].pie(gender_counts.values, labels=gender_counts.index,
               autopct='%1.1f%%', colors=['#4C72B0','#DD8452','#55A868'])
axes[0][1].set_title('Giới tính (EDA: 55/40/5%)')

# 3. Rides/User (NB6)
sns.histplot(df['monthly_rides_history'], bins=20, kde=True,
             ax=axes[0][2], color='coral')
axes[0][2].set_title(f'Rides/tháng\nMean={df["monthly_rides_history"].mean():.1f} (EDA: ~5)')
axes[0][2].axvline(df['monthly_rides_history'].mean(), color='red', linestyle='--')

# 4. Preferred Hour (NB2)
hour_counts = df['preferred_hour'].value_counts().sort_index()
axes[0][3].bar(hour_counts.index, hour_counts.values, color='mediumpurple', alpha=0.8)
axes[0][3].set_title('Giờ gọi xe (NB2: Cao điểm 7-9h, 17-19h)')
axes[0][3].set_xlabel('Giờ trong ngày')
for h in [7, 8, 9, 17, 18, 19]:
    if h in hour_counts.index:
        axes[0][3].axvline(h, color='red', linestyle='--', alpha=0.5)

# 5. Urban/Suburban (NB3)
urban_counts = df['is_urban'].value_counts()
axes[1][0].bar(['Suburban (0)', 'Urban (1)'], urban_counts.values,
               color=['#95a5a6','#2980b9'])
axes[1][0].set_title(f'Vị trí (NB3: Urban ~60%)\nUrban={df["is_urban"].mean()*100:.1f}%')

# 6. Recency Distribution
sns.histplot(df['recency_days'], bins=25, kde=True,
             ax=axes[1][1], color='seagreen')
axes[1][1].set_title(f'Recency (ngày không mở app)\nMean={df["recency_days"].mean():.1f}')

# 7. Customer Segment
seg_counts = df['customer_segment'].value_counts()
axes[1][2].bar(seg_counts.index, seg_counts.values,
               color=['#2ecc71','#3498db','#e74c3c'])
axes[1][2].set_title('Phân khúc Khách hàng')

# 8. Fare Distribution (NB4)
sns.histplot(df['fare_obs'], bins=30, kde=True,
             ax=axes[1][3], color='darkorange')
axes[1][3].set_title(f'Doanh thu (fare_obs)\nMean=${df["fare_obs"].mean():.1f} (NB4: Avg $17.60)')
axes[1][3].axvline(df['fare_obs'].mean(), color='red', linestyle='--')
axes[1][3].axvline(17.60, color='black', linestyle=':', label='EDA target $17.60')
axes[1][3].legend(fontsize=8)

plt.suptitle('Kiểm chứng: Bộ dữ liệu mô phỏng vs. EDA Thực tế', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig('../../reports/data_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Biểu đồ kiểm chứng đã lưu.')

## 4. Kiểm tra Covariate Balance (Chuẩn bị cho Tuần 5 - A/A Test)

Kiểm tra xem Covariates có phân bổ đều giữa nhóm Treatment và Control không.

In [ ]:
def check_balance(data, treatment_col, label):
    covs = ['age', 'is_urban', 'monthly_rides_history', 'recency_days', 'preferred_hour']
    bal = data.groupby(treatment_col)[covs].mean().T
    bal.columns = ['Control (T=0)', 'Treatment (T=1)']
    bal['Chênh lệch (%)'] = (
        (bal['Treatment (T=1)'] - bal['Control (T=0)'])
        / bal['Control (T=0)'] * 100
    ).round(2)
    print(f'\n=== {label} ===')
    display(bal.round(3))

check_balance(df, 'treatment_obs', 'Observational Data (Bị nhiễu)')
check_balance(df, 'treatment_rand', 'Randomized A/B Test')

In [ ]:
# So sánh Naive vs Randomized
naive = (df[df['treatment_obs']==1]['y_obs'].mean()
         - df[df['treatment_obs']==0]['y_obs'].mean())
rand = (df[df['treatment_rand']==1]['y_rand'].mean()
        - df[df['treatment_rand']==0]['y_rand'].mean())

print(f'True ATE (Đáp án thật): {TRUE_ATE:.2f} chuyến')
print(f'Naive Estimate (Bị nhiễu): {naive:.2f} chuyến  (Bias = {naive - TRUE_ATE:+.2f})')
print(f'Randomized Estimate (A/B): {rand:.2f} chuyến  (Bias ≈ {rand - TRUE_ATE:+.2f})')

## 5. Phân tích Doanh thu theo NB4 (OEC cho Tuần 4)

In [ ]:
VOUCHER_DISCOUNT = 0.25  # Giảm giá 25%

treat = df[df['treatment_rand']==1]
ctrl = df[df['treatment_rand']==0]

avg_fare_treat = treat['fare_rand'].mean()
avg_fare_ctrl = ctrl['fare_rand'].mean()
inc_rev = avg_fare_treat - avg_fare_ctrl
voucher_cost = avg_fare_treat * VOUCHER_DISCOUNT
net_inc = inc_rev - voucher_cost
roi = net_inc / voucher_cost

print('=== Economics Metrics (Tuần 4 - OEC) ===')
print(f'Avg Fare Treatment: ${avg_fare_treat:.2f}')
print(f'Avg Fare Control:   ${avg_fare_ctrl:.2f}')
print(f'Incremental Revenue/User: ${inc_rev:.2f}')
print(f'Voucher Cost/User:         ${voucher_cost:.2f}')
print(f'Net Incremental Revenue:   ${net_inc:.2f}')
print(f'True ROI (Net): {roi*100:.1f}%')
print(f'\nSo sánh với EDA (NB4): Avg fare = $17.60, Median = $13.50')
print(f'Avg fare generated:   ${df["fare_rand"].mean():.2f} (Target: ~$17.60)')

## 6. Lưu Dataset cho Tuần 5 & 6

In [ ]:
import os
os.makedirs('../../data/processed', exist_ok=True)
path = '../../data/processed/complex_simulation_data.csv'
df.to_csv(path, index=False)
print(f'Saved: {path}')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')